# Comparison of 2 calibrations 

Comparision of two method of calibration:

- Brucker software 
- Automatic inhouse calibraiton base on mass difference map error

In [ ]:
import spectre_analysis as sp_ana
import calibration as cal
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import os

from pathlib import Path
from scipy.stats import gaussian_kde
from typing import Dict, Union, Optional, Tuple, List

#import datashader as ds
#from datashader import transfer_functions as tf
#import colorcet as cc

## Data extraction

In [2]:
workers = sp_ana.find_workers()

path_std = Path.cwd().parent / "data" / "standards"
path_cal = Path.cwd().parent / "data" / "Standard_Gums_Calibrated_files"

list_std = os.listdir(path_std)
list_cal = os.listdir(path_cal)
list_cal = [i.replace("_Calibrated", "") for i in list_cal]

# take the commun spectra of the both folders
spectra = sorted(list(set(list_std) & set(list_cal)))

In [3]:
# 1) All monomers

monomers_list = ["dHex", "Hex", "HexA", "HexN", "HexNac", "Pent", "Neu", "NeuAc", "NeuGc"]
library = Path.cwd().parent / "data" / "oligo_combinations_clean.csv"
oligo_ref_all = sp_ana.adjust_library(library, monomers_list)
# Transform the mass in ppm and int to optimize the algorithm
for adduct in ['labeled_mass_H+', 'labeled_mass_Na+', 'labeled_mass_K+']:
    oligo_ref_all = sp_ana.normalize_mz_column(
    lf = oligo_ref_all,
    mz_col=adduct,
    new_col=f"{adduct}_int"
)

# 2) Selection of mmonomers

monomers_list = ["Hex", "Pent"]
library = Path.cwd().parent / "data" / "oligo_combinations.csv"
oligo_ref_selection = sp_ana.adjust_library(library, monomers_list)
# Transform the mass in ppm and int to optimize the algorithm
for adduct in ['labeled_mass_H+', 'labeled_mass_Na+', 'labeled_mass_K+']:
    oligo_ref_selection = sp_ana.normalize_mz_column(
    lf = oligo_ref_selection,
    mz_col=adduct,
    new_col=f"{adduct}_int"
)

## Calibration

In [4]:
all_spectrum = []

for spectrum_name in spectra:

    # Extract spectrum
    print(f"Start analysis {spectrum_name}")
    spectrum_file = Path.cwd().parent / "data" / "Standard_Gums_Calibrated_files" / spectrum_name
    spectrum_cal = sp_ana.read_spectrum(spectrum_file)
    spectrum_file = Path.cwd().parent / "data" / "standards" / spectrum_name
    spectrum_std = sp_ana.read_spectrum(spectrum_file)
    
    # Extract peaks 
    print("Peak extraction")
    peaks = sp_ana.extract_and_detect_peaks(
        spectrum_data=spectrum_std,
        window=500,
        dist_min=0,
        prominence=3e5,
        n_workers=workers
    )

    # Normalized
    print("Peak normalization")
    peaks = sp_ana.normalize_mz_column(
        lf = peaks,
        mz_col="m/z",
        new_col="mz_int"
    )
    
    # Build mass difference map
    print('Start map difference map')
    mdim_all_mono = cal.build_mass_difference_map(peaks, oligo_ref_all, threshold_ppm=500)
    mdim_selection_mono = cal.build_mass_difference_map(peaks, oligo_ref_selection, threshold_ppm=500)

    print("Calculate the error table")
    error_table_all = cal.calculate_error_table(mdim_all_mono)
    error_table_selection = cal.calculate_error_table(mdim_selection_mono)

    print("New calibration")
    corrected_spectrum_all = cal.interpolate_corrections(spectrum_std, error_table_all, new_col_name="all_mono")
    corrected_spectrum_selection = cal.interpolate_corrections(spectrum_std, error_table_selection, new_col_name="selected_mono")

    # Join
    print('Join the spectrum')
    global_data = (corrected_spectrum_all
                   .join(
                        other=corrected_spectrum_selection,
                        on="m/z",
                        how="inner",)
                    .with_columns(
                        pl.lit(spectrum_name).alias("Spectrum_Name")
                    ))

    all_spectrum.append(global_data)

print("final concatenation")
final_global_data = pl.concat(all_spectrum)


    

Start analysis 18052022-arabinogalactane-0.01mg_0_0_L13_000001.csv
Peak extraction
Peak normalization
Start map difference map
Calculate the error table
New calibration
Join the spectrum
Start analysis 18052022-arabinogalactane-0.01mg_0_0_L13_000002.csv
Peak extraction
Peak normalization
Start map difference map
Calculate the error table
New calibration


: 